# Fractal Basins Lab — Colab edition

Probe the fractal convergence landscape of a looped reasoning model (EqR, ~27M params)
on a free T4. Implements the basin protocol from *Fractal basins trap latent reasoning*
(Lai et al. 2026) via the [loopscape](https://github.com/GilpinLab/loopscape) toolkit.

**Runtime → Change runtime type → T4 GPU** before running.

Cells: install → basin slices (stills + drift animation) → state-capture (crystallization animation) → display & download.

In [ ]:
# 1. Clone the lab + install the probing toolkit (~2 min)
!git clone --depth 1 https://github.com/terrafying/fractal-basins-lab.git
%cd /content/fractal-basins-lab
!pip -q install "loopscape @ git+https://github.com/GilpinLab/loopscape"
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2. Basin slices: 2 puzzles x 1 seed at 64^2 (16k conditions, ~10-20 min on T4)
#    Scale up with FB_RES=128/200 and FB_SEEDS=0,1,2 for paper-resolution runs.
import os
os.environ['FB_RES'] = '64'
os.environ['FB_SEEDS'] = '0'
!python exp01_sudoku_slices.py

In [ ]:
# 3. Render stills + slow-drift animations (smoothed; fractal filaments visible)
!python render_basins.py
!python render_basins.py --drift
from IPython.display import Image, display
for f in sorted(__import__('pathlib').Path('runs/figs').glob('*_seed*.png')):
    print(f); display(Image(str(f), width=420))

In [ ]:
# 4. State capture on the hard puzzle + crystallization animation
#    (per-loop disagreement with final solution; watch the basin geometry form)
import os
os.environ['FB_RES'] = '48'
!python exp02_state_capture.py
!python anim_crystallization.py

In [ ]:
# 5. Display animations inline + package results for download
from IPython.display import HTML
from pathlib import Path
for f in sorted(Path('runs/figs').glob('*.mp4')):
    print(f)
    display(HTML(f'<video width=480 controls src="{f}"></video>'))
!cd runs && zip -qr /content/fractal_basins_results.zip figs exp01 exp02
print('download /content/fractal_basins_results.zip from the file browser')

## Going further

- Paper resolution: `FB_RES=200 FB_SEEDS=0,1,2` (~4 h/slice on T4; run overnight or rent a 3090 on vast.ai for ~$0.20/hr).
- Second architecture: swap `get_solver("eqr")` for `"fprm"` in the exp scripts (richer transients, 200-iter budget).
- Metrics: `basin_entropy`, `uncertainty_exponent`, `fli_discrete` from `loopscape.metrics`.
- Protocol + analogy table: see AGENTS.md.